# Title

Angos and Tan

BSDSBA 2028

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

### Data Download

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("usdot/flight-delays", output_dir="./data")

# print("Path to dataset files:", path)

# OVERVIEW

This mini-project analyzes airline operations using the **U.S. Department of Transportation (DOT) 2015 Flight Delays and Cancellations dataset**. The project investigates how airport congestion, airline scheduling practices, and traffic density contribute to flight delays. In addition to identifying structural causes of delays, the project aims to forecast flight delay duration based on operational conditions, including airport traffic, route characteristics, and schedule realism.

Airline delays are often attributed to weather or operational disruptions, but underlying causes may include unrealistic scheduling and airport capacity constraints. By analyzing patterns in flight schedules, traffic density, and delay distributions, this project aims to identify systemic patterns that contribute to delays and congestion.

Understanding these factors has practical applications for airlines, airports, and passengers. Airlines can improve schedule planning, airports can identify congestion thresholds, and travelers can better understand which routes or airports are more reliable.


## GOALS

1. Analyze patterns in flight delays across airlines, routes, airports, and time of day.

2. Identify congestion thresholds at which airport traffic density results in significant increases in delay.

3. Forecast expected delay duration for flights using operational features such as airport traffic, route reliability, and schedule realism.

4. Evaluate whether airline schedules are realistic by comparing scheduled flight time with actual flight durations.


## Basic

TODO:
- 
- See whether delays affect a planes next flight

## General Data Exploration

In [ ]:
# Load Data
airlines = pd.read_csv("data/airlines.csv")
airports = pd.read_csv("data/airports.csv")
flights = pd.read_csv("data/flights.csv", low_memory=False)

In [63]:
#fix from https://github.com/lezandar/flights
aircode1 = pd.read_csv('data/L_AIRPORT.csv')
aircode2 = pd.read_csv('data/L_AIRPORT_ID.csv')

# Format the airport codes
aircode1 = aircode1.reset_index()
aircode2 = aircode2.reset_index()

In [64]:
aircodes = pd.merge(aircode1,aircode2,on='Description')
aircode_dict = dict(zip(aircodes['Code_y'].astype(str),aircodes['Code_x']))

In [65]:
# Make sure all Origin and departing airports are strings
flights['ORIGIN_AIRPORT'] = flights['ORIGIN_AIRPORT'].values.astype(str)
flights['DESTINATION_AIRPORT'] = flights['DESTINATION_AIRPORT'].values.astype(str)

In [ ]:
for i in range(len(flights)):
    if len(flights['ORIGIN_AIRPORT'][i]) != 3:
        print(flights['ORIGIN_AIRPORT'][i])
        to_replace = flights['ORIGIN_AIRPORT'][i]
        value = aircode_dict[flights['ORIGIN_AIRPORT'][i]]
        flights = flights.replace(to_replace, value)

ANC
LAX
SFO
LAX
SEA
SFO
LAS
LAX
SFO
LAS
DEN
LAS
LAX
SLC
SEA
ANC
ANC
SFO
ANC
PDX
LAS
SEA
LAS
LAX
LAS
LAX
FAI
MSP
LAS
DEN
PHX
PHX
ANC
SLC
LAS
LAS
ANC
SJU
ANC
SJU
PBG
PHX
PHX
ANC
IAG
SJU
PHX
ANC
PSE
BQN
BQN
SJU
SJU
BQN
PSE
SJU
ORD
GEG
HNL
ONT
HNL
ANC
HNL
ORD
MCO
BOS
HIB
ABR
MAF
SEA
DFW
BOS
MKE
PDX
IAH
BNA
BRO
VPS
BOI
BJI
HNL
BNA
SGF
PHL
PDX
SEA
DEN
SBN
PHL
ORD
RDD
EUG
SFO
ORD
IAD
BUF
PWM
JFK
CRP
PIA
ORD
FAT
SMF
AUS
BOS
ORD
PHX
MCI
BOS
ATL
JAX
MFR
SLC
IDA
MSN
MKE
GEG
SEA
DCA
SAT
JFK
BOS
BOS
JFK
DFW
CHS
MKE
MSP
CHS
DEN
SBA
SMX
HNL
BOS
PDX
IND
LAX
CLE
BOS
DEN
MCI
AUS
LAX
GSP
MCO
MCI
BDL
ABI
RIC
BFL
SAT
OMA
RDM
FLL
CID
SFO
MCO
TPA
SEA
BOS
SYR
ROC
TYR
MCI
LAN
BUF
XNA
GSO
ORD
LAX
EWR
ONT
MSP
PBI
PHX
RSW
EWR
OMA
SMF
OAK
PVD
RNO
PIT
DEN
BOS
LAS
ABQ
PHX
PIT
PVD
SFO
SFO
MCO
PDX
BOS
SFO
SAT
BOS
SEA
LAX
IAH
MCO
SFO
DFW
MIA
SLC
DCA
MIA
BWI
DFW
LGA
DEN
MCI
MCO
SFO
TUL
LGA
LIT
MIA
LAS
PDX
TPA
SMF
MSY
LAX
OMA
OKC
SEA
FAI
SEA
SEA
PDX
LAX
PHX
DEN
GEG
RSW
PBI
JFK
TPA
FLL
OAK
LGA
JFK
EWR
JFK
PSE
MSY
AUS
BQN


KeyboardInterrupt: 

In [ ]:
for i in range(len(flights)):
    if len(flights['DESTINATION_AIRPORT'][i]) != 3:
        to_replace = flights['DESTINATION_AIRPORT'][i]
        value = aircode_dict[flights['DESTINATION_AIRPORT'][i]]
        flights = flights.replace(to_replace, value)

KeyboardInterrupt: 

In [ ]:
display(flights.head())
print(flights.info())
print(flights.shape)


### Flights Composition:

In [ ]:
cancelled_flights = (
    (flights.iloc[:, flights.columns.get_loc('CANCELLED')] == 1).sum()
    / len(flights))

diverted_flights = (
    (flights.iloc[:, flights.columns.get_loc('DIVERTED')] == 1).sum()
    / len(flights))

success_flights = 1 - cancelled_flights + diverted_flights

In [ ]:
success_flights_df = flights.loc[
    (
        (flights.iloc[:, flights.columns.get_loc("DIVERTED")].astype(int) == 0)
        & (flights.iloc[:, flights.columns.get_loc("CANCELLED")].astype(int) == 0)
    )
]

len_success_flights = len(success_flights_df)

In [ ]:
d_delay_flights = (
    success_flights_df.iloc[:, 
    success_flights_df.columns.get_loc('DEPARTURE_DELAY')] > 0
    ).sum() / len_success_flights

a_delay_flights = (
    success_flights_df.iloc[:, 
    success_flights_df.columns.get_loc('ARRIVAL_DELAY')] > 0
    ).sum() / len_success_flights

both_delay_flights = (
    (
        (success_flights_df['DEPARTURE_DELAY'] > 0)
        & (success_flights_df['ARRIVAL_DELAY'] > 0)
    ).sum()
) / len_success_flights

In [ ]:
print(f"Cancelled flights: {cancelled_flights*100:.2f}%")
print(f"Diverted flights: {diverted_flights*100:.2f}%")
print(f"Successful flights: {success_flights*100:.2f}%")

In [ ]:
print(f"Delayed Departures: {d_delay_flights*100:.2f}% of successful flights")
print(f"Delayed Arrivals: {a_delay_flights*100:.2f}% of successful flights")
print(f"Delayed Departures with delayed arrivals: {both_delay_flights*100:.2f}% of successful flights")


### CLEANING DATA NEEDS
- Some airports are numbers instead of IATA codes, no reference table in dataset, unable to find elsewhere

Solution: temporarily ignore all october data

- Add Departure Hour

In [ ]:
flights["DEPARTURE_HOUR"] = flights["DEPARTURE_TIME"] // 100

In [ ]:
flights[flights['MONTH'] == 10][[
    'MONTH',
    'ORIGIN_AIRPORT',
    'DESTINATION_AIRPORT']].tail()

In [ ]:
sns.histplot(flights["ARRIVAL_DELAY"], bins=100)
plt.xlim(-50, 200)
plt.title("Distribution of Arrival Delays")
plt.xlabel("Arrival Delay (minutes)")
plt.show()

In [ ]:
delay_by_airline = flights.groupby("AIRLINE")["ARRIVAL_DELAY"].mean().sort_values()

delay_by_airline.plot(kind="bar")
plt.title("Average Delay by Airline")
plt.ylabel("Average Delay (minutes)")
plt.show()

In [ ]:
flights["DepHour"] = flights["SCHEDULED_DEPARTURE"] // 100

delay_by_hour = flights.groupby("DepHour")["ARRIVAL_DELAY"].mean()

delay_by_hour.plot()
plt.title("Average Delay by Departure Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Delay")
plt.show()

In [ ]:
traffic = flights.groupby(["ORIGIN_AIRPORT","DepHour"]).size()

traffic_by_hour = traffic.groupby("DepHour").mean()

traffic_by_hour.plot()
plt.title("Average Flights per Airport by Hour")
plt.ylabel("Flights")
plt.show()

In [ ]:
flights["ON_TIME"] = flights["ARRIVAL_DELAY"].fillna(float('inf')) <= 10

route_reliability = (
    flights.groupby(["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"])["ON_TIME"]
    .mean()
    .sort_values()
)

print(route_reliability.head(10))

In [ ]:

# --- Filter to first week, remove cancelled/diverted ---
week = flights[
    (flights['MONTH'] == 1) &
    (flights['DAY'].between(1, 7)) &
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0) &
    flights['DEPARTURE_HOUR'].notna() &
    flights['ARRIVAL_DELAY'].notna()
].copy()
# --- Delay color/width bins ---
# Blue = on time, yellow → orange → red → dark red = increasingly late
DELAY_BINS   = [-np.inf, 0, 15, 30, 60, np.inf]
DELAY_COLORS = [
    'rgba(100,149,237,0.5)',  # blue      — on time / early
    'rgba(255,220,50,0.6)',   # yellow    — 1–15 min
    'rgba(255,140,0,0.7)',    # orange    — 15–30 min
    'rgba(220,40,40,0.8)',    # red       — 30–60 min
    'rgba(139,0,0,0.9)',      # dark red  — 60+ min
]
DELAY_WIDTHS = [0.5, 1, 2, 3, 4.5]


In [ ]:
def build_edge_traces(hour_flights):
    # Always initialize all 6 traces as empty
    on_time_lats, on_time_lons = [], []
    bin_lats  = {1: [], 2: [], 3: [], 4: [], 6: []}
    bin_lons  = {1: [], 2: [], 3: [], 4: [], 6: []}

    if not hour_flights.empty:
        routes = (
            hour_flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .agg(COUNT=('FLIGHT_NUMBER', 'count'), AVG_DELAY=('ARRIVAL_DELAY', 'mean'))
            .reset_index()
        )

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='ORIGIN_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'orig_lat', 'LONGITUDE': 'orig_lon'}).drop(columns='IATA_CODE')

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='DESTINATION_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'dest_lat', 'LONGITUDE': 'dest_lon'}).drop(columns='IATA_CODE')

        routes['DELAYED_COUNT'] = (
            hour_flights[hour_flights['ARRIVAL_DELAY'] > 5]
            .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .size()
            .reindex(pd.MultiIndex.from_frame(routes[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']]))
            .fillna(0)
            .values
        )

        on_time = routes[routes['DELAYED_COUNT'] == 0]
        delayed = routes[routes['DELAYED_COUNT'] > 0].copy()

        for _, row in on_time.iterrows():
            on_time_lats += [row['orig_lat'], row['dest_lat'], None]
            on_time_lons += [row['orig_lon'], row['dest_lon'], None]

        if not delayed.empty:
            delayed['width'] = pd.cut(
                delayed['DELAYED_COUNT'], bins=5,
                labels=[1, 2, 3, 4, 6],
                duplicates='drop'
            ).astype(float).fillna(1)

            for _, row in delayed.iterrows():
                w = row['width']
                bin_lats[w] += [row['orig_lat'], row['dest_lat'], None]
                bin_lons[w] += [row['orig_lon'], row['dest_lon'], None]


    traces = [
        go.Scattermap(lat=on_time_lats, lon=on_time_lons, mode='lines',
                      line=dict(width=0.5, color='rgba(100,149,237,0.2)'),
                      hoverinfo='none', showlegend=False),
    ]
    for w, color in zip([1, 2, 3, 4, 6], [
        'rgba(255,220,50,0.6)',
        'rgba(255,160,0,0.7)',
        'rgba(220,80,40,0.8)',
        'rgba(200,20,20,0.85)',
        'rgba(139,0,0,0.9)',
    ]):
        traces.append(go.Scattermap(
            lat=bin_lats[w], lon=bin_lons[w], mode='lines',
            line=dict(width=w, color=color),
            hoverinfo='none', showlegend=False,
        ))

    return traces  

In [ ]:
active_iata = set(week['ORIGIN_AIRPORT']) | set(week['DESTINATION_AIRPORT'])
active_airports = airports[airports['IATA_CODE'].isin(active_iata)]

node_trace = go.Scattermap(
    lat=active_airports['LATITUDE'],
    lon=active_airports['LONGITUDE'],
    mode='markers',
    marker=dict(size=5, color='white', opacity=0.8),
    text=active_airports['IATA_CODE'] + ' — ' + active_airports['CITY'],
    hoverinfo='text',
    showlegend=False,
)

In [ ]:
# --- Build all 168 frames (7 days × 24 hours) ---
frames = []
slider_steps = []

for day in range(1,8):
    for hour in range(24):
        label = f"Jan {day:02d}  {hour:02d}:00"
        slice_ = week[(week['DAY'] == day) & (week['DEPARTURE_HOUR'] == hour)]
        edge_traces = build_edge_traces(slice_)

        frames.append(go.Frame(
            data=edge_traces + [node_trace],
            name=label
        ))
        slider_steps.append(dict(
            method='animate',
            args=[[label], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
            label=f"D{day} {hour:02d}h"
        ))

In [ ]:
# --- Initial display (Jan 1, 08:00 — usually busiest) ---
init_traces = build_edge_traces(week[(week['DAY'] == 1) & (week['DEPARTURE_HOUR'] == 8)])

fig = go.Figure(
    data=init_traces + [node_trace],
    frames=frames
)

fig.update_layout(
    map=dict(style='carto-darkmatter', center=dict(lat=39, lon=-98), zoom=3),
    margin=dict(r=0, t=50, l=0, b=0),
    title=dict(text='US Flight Delays — Jan 1–7 2015 (Hourly)', font=dict(color='white')),
    paper_bgcolor='#1a1a2e',
    updatemenus=[dict(
        type='buttons', showactive=False, y=1.08, x=0.0,
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(mode='immediate')]),
        ]
    )],
    sliders=[dict(
        steps=slider_steps,
        currentvalue=dict(prefix='Time: ', font=dict(size=13, color='white')),
        font=dict(color='white'),
        len=0.95, x=0.02,
    )]
)

fig.show()

In [ ]:
flights[((flights["MONTH"] == 1)&(flights["DAY"] == 3)&(flights["DEPARTURE_HOUR"]==1))]

In [ ]:
print(routes['COUNT'].describe())
print(routes['width'].value_counts())
print(routes[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'COUNT', 'width']].head(10))

In [ ]:
filtered_flights = flights[flights['CANCELLED'] == 0]

airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT','hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
           .round()
)

In [ ]:
airports

In [ ]:
valid_flights = flights[
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0)
]

route_df = (
    valid_flights
    .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], as_index=False)
    .agg(AVE_FLIGHT_TIME=('ELAPSED_TIME', 'mean'))
)

## Intermediate

## Advanced